# TP2 - Classification binaire, sous-gradient et regularisation

Ce notebook realise les questions demandees dans la partie TP2 du mini-projet, en restant coherent avec le cadre experimental retenu pour CelebA.


## Objectif TP2

Le TP2 demande d'etudier la classification binaire dans un cadre **non differentiable**.

Dans ce notebook, on couvre:

- **TP2-P**: definition des deux classes, de la sortie du CNN et de la regle de decision
- **TP2-SG**: perte hinge et sous-gradient
- **TP2-D**: choix de direction `d_t = -g_t` et comparaison avec une mauvaise direction
- **TP2-LS**: line search avec Armijo, Goldstein, Wolfe et une variante adaptative
- **TP2-VAL**: validation, detection d'underfitting/overfitting, regularisation


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from torch import optim
from torch.utils.data import DataLoader, Subset

from data_loader import create_dataloaders, resolve_default_paths
from experiment_spec import EXPERIMENTAL_FRAME
from train_common import BinaryHingeLoss, build_model, evaluate_classification
from utils import adaptive_line_search, armijo, goldstein, plot_multi_losses, wolfe

plt.style.use('ggplot')
try:
    get_ipython().run_line_magic('matplotlib', 'inline')
except NameError:
    pass

torch.manual_seed(42)
np.random.seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
frame = EXPERIMENTAL_FRAME['classification']
default_paths = resolve_default_paths()

print('Device:', device)
print('Classification frame:', frame)


## TP2-P - Definition de la classification binaire par CNN

Nous retenons les deux classes suivantes:

- classe positive `+1`: visage souriant (`Smiling = 1`)
- classe negative `-1`: visage non souriant (`Smiling = -1`)

Le CNN produit un **score scalaire brut** `s_theta(x)`.

La regle de decision binaire est:

`y_hat = sign(s_theta(x))`

Ce choix est aligne avec le PDF du mini-projet et permet d'utiliser une perte hinge non differentiable.


In [ ]:
cls_loaders, cls_sizes = create_dataloaders(
    img_dir=default_paths['image_dir'],
    attr_file=default_paths['attr_file'],
    partition_file=default_paths['partition_file'],
    target_type='classification',
    target_column=frame['target_column'],
    classification_label_scheme=frame['label_scheme'],
    batch_size=32,
    image_size=64,
    num_workers=0,
    seed=42,
    val_ratio=0.15,
    test_ratio=0.15,
)

print('Classification split sizes:', cls_sizes)

train_dataset = cls_loaders['train'].dataset
if hasattr(train_dataset, 'dataset') and hasattr(train_dataset, 'indices'):
    base_df = train_dataset.dataset.metadata.iloc[train_dataset.indices].copy()
else:
    base_df = train_dataset.metadata.copy()

label_counts = pd.Series(base_df['target']).value_counts().sort_index()
display(label_counts.rename('count').to_frame())

plt.figure(figsize=(5, 4))
sns.countplot(x=base_df['target'].astype(int), color='steelblue')
plt.title('Binary class distribution for TP2')
plt.xlabel('Signed label')
plt.ylabel('Count')
plt.show()


## TP2-SG - Fonction de cout et sous-gradient

On utilise la perte hinge binaire:

`L(x, y; theta) = max(0, 1 - y s_theta(x))`

avec `y` dans `{-1, +1}`.

Cette perte n'est pas differentiable au point ou `1 - y s_theta(x) = 0`.
Le gradient ordinaire n'est donc pas defini partout. On travaille alors avec un **sous-gradient**.


In [ ]:
criterion_cls = BinaryHingeLoss()
cls_model = build_model(EXPERIMENTAL_FRAME['models']['CNN1'], task='classification').to(device)
optimizer_cls = optim.SGD(cls_model.parameters(), lr=1e-3)

inputs_batch, targets_batch = next(iter(cls_loaders['train']))
inputs_batch = inputs_batch.to(device)
targets_batch = targets_batch.to(device).view(-1)

cls_model.train()
optimizer_cls.zero_grad()
scores = cls_model(inputs_batch).view(-1)
loss = criterion_cls(scores, targets_batch)
loss.backward()

margins = 1.0 - targets_batch * scores.detach()
active_mask = (margins > 0).float()
manual_subgrad_scores = (-targets_batch * active_mask) / targets_batch.numel()

print(f'Hinge loss on one batch: {loss.item():.4f}')
print(f'Active constraints: {int(active_mask.sum().item())} / {len(active_mask)}')
print('First 10 scores:')
display(pd.DataFrame({
    'target': targets_batch.detach().cpu().numpy()[:10],
    'score': scores.detach().cpu().numpy()[:10],
    'margin': margins.detach().cpu().numpy()[:10],
    'manual_subgrad_on_score': manual_subgrad_scores.detach().cpu().numpy()[:10],
}))

param_grad_norms = {name: float(param.grad.norm().item()) for name, param in cls_model.named_parameters() if param.grad is not None}
display(pd.Series(param_grad_norms).sort_values(ascending=False).head(8).rename('gradient_norm'))


### Interpretation

- Si `1 - y s_theta(x) > 0`, l'exemple viole la marge et contribue au sous-gradient.
- Si `1 - y s_theta(x) < 0`, l'exemple est correctement classe avec marge suffisante et son sous-gradient est nul.
- Au bord `1 - y s_theta(x) = 0`, la fonction n'est pas differentiable: c'est precisement pourquoi le **gradient ordinaire n'est pas suffisant**.


## TP2-D - Choix de direction

Dans un schema de descente du premier ordre, on prend naturellement:

`d_t = -g_t`

ou `-g_t` pour un sous-gradient `g_t`.

Cette direction est la direction opposee a la pente locale, donc une direction de descente. Une mauvaise direction naturelle a comparer est `+g_t`, qui tend au contraire a augmenter la perte locale.


In [ ]:
selected_params = [cls_model.fc_out.weight, cls_model.fc_out.bias]
selected_shapes = [tuple(param.shape) for param in selected_params]
selected_sizes = [param.numel() for param in selected_params]

theta0 = np.concatenate([param.detach().cpu().numpy().reshape(-1) for param in selected_params]).astype(np.float64)
g0 = np.concatenate([param.grad.detach().cpu().numpy().reshape(-1) for param in selected_params]).astype(np.float64)

def assign_last_layer(theta_np):
    offset = 0
    with torch.no_grad():
        for param, shape, size in zip(selected_params, selected_shapes, selected_sizes):
            values = theta_np[offset: offset + size]
            tensor = torch.tensor(values, dtype=param.dtype, device=param.device).view(shape)
            param.copy_(tensor)
            offset += size

def batch_loss_last_layer(theta_np):
    assign_last_layer(theta_np)
    with torch.no_grad():
        scores_local = cls_model(inputs_batch).view(-1)
        return float(criterion_cls(scores_local, targets_batch).item())

def batch_subgrad_last_layer(theta_np):
    assign_last_layer(theta_np)
    cls_model.zero_grad()
    loss_local = criterion_cls(cls_model(inputs_batch).view(-1), targets_batch)
    loss_local.backward()
    return np.concatenate([param.grad.detach().cpu().numpy().reshape(-1) for param in selected_params]).astype(np.float64)

good_direction = -g0
bad_direction = +g0
alpha_test = 1e-2
loss_at_theta = batch_loss_last_layer(theta0)
loss_good = batch_loss_last_layer(theta0 + alpha_test * good_direction)
loss_bad = batch_loss_last_layer(theta0 + alpha_test * bad_direction)
assign_last_layer(theta0)

print(f'Loss at current point:        {loss_at_theta:.6f}')
print(f'Loss after step with -g_t:    {loss_good:.6f}')
print(f'Loss after step with +g_t:    {loss_bad:.6f}')


## TP2-LS - Line Search en cadre non differentiable

On applique les regles de line search a la perte hinge sur la derniere couche du CNN. En pratique, la fonction est non differentiable mais piecewise differentiable, ce qui permet d'utiliser un sous-gradient presque partout.


In [ ]:
armijo_alpha = armijo(batch_loss_last_layer, theta0, good_direction, g0)
goldstein_alpha = goldstein(batch_loss_last_layer, theta0, good_direction, g0)
wolfe_alpha = wolfe(batch_loss_last_layer, batch_subgrad_last_layer, theta0, good_direction, g0)
adaptive_alpha, adaptive_success = adaptive_line_search(batch_loss_last_layer, theta0, good_direction, alpha=1.0)
assign_last_layer(theta0)

line_search_df = pd.DataFrame({
    'Method': ['Armijo', 'Goldstein', 'Wolfe', 'Adaptive'],
    'Step size': [armijo_alpha, goldstein_alpha, wolfe_alpha, adaptive_alpha],
    'Accepted/Success': [True, True, True, adaptive_success],
})
display(line_search_df)


## TP2-VAL - Validation et regularisation

On utilise les splits `train/validation/test` deja construits.
Pour discuter underfitting et overfitting, on compare les erreurs train/validation pour plusieurs configurations de regularisation sur la classification binaire.


In [ ]:
def make_small_loader(loader, limit, shuffle=False):
    subset_indices = list(range(min(limit, len(loader.dataset))))
    subset = Subset(loader.dataset, subset_indices)
    return DataLoader(subset, batch_size=loader.batch_size, shuffle=shuffle, num_workers=0)

small_train_loader = make_small_loader(cls_loaders['train'], limit=512, shuffle=True)
small_val_loader = make_small_loader(cls_loaders['val'], limit=256, shuffle=False)

def fit_classification_history(model_name, epochs=4, lr=1e-3, weight_decay=0.0, l1_lambda=0.0):
    model = build_model(model_name, task='classification').to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = BinaryHingeLoss()
    history = {'train_loss': [], 'val_loss': [], 'train_f1': [], 'val_f1': []}

    for _ in range(epochs):
        model.train()
        for inputs, targets in small_train_loader:
            inputs = inputs.to(device)
            targets = targets.to(device).view(-1)
            optimizer.zero_grad()
            outputs = model(inputs).view(-1)
            loss_local = criterion(outputs, targets)
            if l1_lambda > 0:
                loss_local = loss_local + l1_lambda * sum(param.abs().sum() for param in model.parameters())
            loss_local.backward()
            optimizer.step()

        train_metrics = evaluate_classification(model, small_train_loader, criterion, device)
        val_metrics = evaluate_classification(model, small_val_loader, criterion, device)
        history['train_loss'].append(train_metrics['loss'])
        history['val_loss'].append(val_metrics['loss'])
        history['train_f1'].append(train_metrics['f1'])
        history['val_f1'].append(val_metrics['f1'])

    return history

base_history = fit_classification_history('simple', weight_decay=0.0, l1_lambda=0.0)
l2_history = fit_classification_history('simple', weight_decay=1e-4, l1_lambda=0.0)
l1_history = fit_classification_history('simple', weight_decay=0.0, l1_lambda=1e-6)

plot_multi_losses({
    'Base train hinge': base_history['train_loss'],
    'Base val hinge': base_history['val_loss'],
    'L2 val hinge': l2_history['val_loss'],
    'L1 val hinge': l1_history['val_loss'],
}, title='TP2 validation and regularisation on hinge loss')
plt.show()

validation_summary = pd.DataFrame({
    'Setting': ['No regularisation', 'L2 = 1e-4', 'L1 = 1e-6'],
    'Train F1': [base_history['train_f1'][-1], l2_history['train_f1'][-1], l1_history['train_f1'][-1]],
    'Validation F1': [base_history['val_f1'][-1], l2_history['val_f1'][-1], l1_history['val_f1'][-1]],
    'Train hinge loss': [base_history['train_loss'][-1], l2_history['train_loss'][-1], l1_history['train_loss'][-1]],
    'Validation hinge loss': [base_history['val_loss'][-1], l2_history['val_loss'][-1], l1_history['val_loss'][-1]],
})
validation_summary['Generalisation gap'] = validation_summary['Validation hinge loss'] - validation_summary['Train hinge loss']
display(validation_summary)


### Lecture underfitting / overfitting

- **Underfitting**: train loss elevee et validation loss elevee.
- **Overfitting**: train loss faible mais validation loss sensiblement plus forte.
- **Regularisation utile**: une regularisation est interessante si elle reduit la perte de validation ou ameliore le F1 de validation, meme si la performance train baisse legerement.


## Synthese TP2

Ce notebook repond aux points TP2 du sujet:

- **TP2-P.1**: classes choisies = `Smiling` contre `Not Smiling`
- **TP2-P.2**: sortie du CNN = score scalaire brut
- **TP2-P.3**: regle de decision = `sign(score)`
- **TP2-SG.1**: perte hinge non differentiable
- **TP2-SG.2**: sous-gradient calcule et interprete
- **TP2-SG.3**: explication de la limite du gradient ordinaire
- **TP2-D.1 a TP2-D.3**: justification de `d_t = -g_t` et comparaison avec `+g_t`
- **TP2-LS.1 a TP2-LS.4**: Armijo, Goldstein, Wolfe et variante adaptative
- **TP2-VAL.1 a TP2-VAL.3**: validation et regularisation

Le notebook est donc le support naturel du chapitre TP2 dans le rapport final, a condition d'etre execute avec le dossier `data/` present localement.
